# 05 — Fresh Clean End-to-End Run (Week 4 / Nov 4 task)

**Task (Excel Week 4, 04-Nov):** *Clean integrated execution — run complete workflow
from fresh synthetic inputs and confirm expected KPI tables. Acceptance: clean
end-to-end execution with 4 datasets, 3 KPIs and DQ summary; logs captured.*

**Design (baseline is never touched):**
* fresh synthetic inputs are staged into the **separate UC volume**
  `workspace.landing.fresh_inputs` (baseline `raw_files` volume is untouched);
* the run executes in **isolated schemas** `workspace.fresh_bronze / fresh_silver /
  fresh_gold`, so the production `bronze / silver / gold` schemas are untouched;
* every stage is logged to `workspace.freshtest.pipeline_run_log` (same shape as
  the production `workspace.gold.pipeline_run_log`).

**Expected results (known-good baseline):**

| Stage | Expected |
| --- | --- |
| Bronze | studies 6 + sites 26 + subjects 280 + visits 1492 = **1804** rows |
| Silver DQ gate | **7/7 PASS**, 0 violations |
| Silver | 267 subjects + 5 quarantined, 1463 visits + 29 quarantined (total upserted **1796**) |
| Gold | study_kpi_summary **6** + site_kpi_summary **26** = **32** KPI rows |
| KPI fingerprints | byte-identical to production gold (shared columns): study `ea64730bc6effd53`, site `3ddfe6e7889e09e6` |

Run top-to-bottom; every execution is idempotent (fresh schemas are reset at
the start of the run cell).

In [0]:
# --- Bootstrap: make the repo's src/ package importable + config ---
import datetime
import sys

from databricks.sdk.runtime import dbutils

try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().value()
    _root = "/" + "/".join(_nb.strip("/").split("/")[:-2])  # .../notebooks/05_... -> repo root
except Exception:
    _root = "/Users/sreekanthreddy2000@gmail.com/clinical-trial-dq-pip"
sys.path.insert(0, "/Workspace" + _root)

from src.transforms import (
    BUSINESS_COLUMNS, build_site_kpi_summary, build_study_kpi_summary,
    independent_kpi_queries, new_rows_only, read_csv_all_strings,
    transform_sites, transform_studies, transform_subjects, transform_visits,
)
from src.dq_checks import all_passed, run_dq_checks, summarize

# SEPARATE volume + SEPARATE schemas + SEPARATE run log: baseline untouched
FRESH_LANDING = "/Volumes/workspace/landing/fresh_inputs"
BRONZE, SILVER, GOLD = "workspace.fresh_bronze", "workspace.fresh_silver", "workspace.fresh_gold"
RUN_LOG = "workspace.freshtest.pipeline_run_log"
RUN_ID = "fresh-e2e-" + datetime.datetime.now(datetime.UTC).strftime("%Y%m%d-%H%M%S")
DATASETS = [("studies", "studies.csv"), ("sites", "sites.csv"),
            ("subjects", "subjects.csv"), ("visits", "visits.csv")]

# Stage the fresh synthetic inputs (clean copies of the version-controlled dataset
# in data/) into the separate volume. Overwrite refreshes mtimes so the ingest is
# exercised on every run; the anti-join keeps it idempotent anyway.
for _f in [t for _, t in DATASETS]:
    with open(f"/Workspace{_root}/data/{_f}") as fh:
        dbutils.fs.put(f"{FRESH_LANDING}/{_f}", fh.read(), overwrite=True)
print(f"Fresh inputs staged in {FRESH_LANDING} | run_id={RUN_ID}")
display(dbutils.fs.ls(FRESH_LANDING))

In [0]:
# --- Stage harness: the same logic as 04_orchestrate, parameterized for the
# fresh (isolated) schemas. Includes the DEF-02 hardening (skip only when the
# bronze table also has data) and benefits from the DEF-01 header validation
# inside read_csv_all_strings (fixed Nov 5, see NEGATIVE_TESTS.md).

def _log_stage(stage_name, start_ts, status, records=None, error=None):
    end_ts = datetime.datetime.now(datetime.UTC)
    err = None if error is None else f"{stage_name} failed: {type(error).__name__}: {error}"
    spark.createDataFrame([(RUN_ID, stage_name, start_ts, end_ts, status, records, err)],
        "run_id string, stage_name string, start_time timestamp, end_time timestamp, "
        "status string, records_processed bigint, error_message string"
    ).createOrReplaceTempView("_flog_row")
    spark.sql(f"""MERGE INTO {RUN_LOG} t USING _flog_row s
                 ON t.run_id = s.run_id AND t.stage_name = s.stage_name
                 WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *""")
    msg = f"[RUN LOG] run_id={RUN_ID} | stage={stage_name} | status={status}"
    if records is not None:
        msg += f" | records={records}"
    if err:
        msg += f" | {err}"
    print(msg)

def _upsert(df, schema, table, key, delete_not_matched_by_source=False):
    df.createOrReplaceTempView("v_src")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {schema}.{table} USING DELTA "
             f"AS SELECT * FROM v_src WHERE 1=0")
    extra = " WHEN NOT MATCHED BY SOURCE THEN DELETE" if delete_not_matched_by_source else ""
    spark.sql(f"""MERGE INTO {schema}.{table} t USING v_src s ON t.{key} = s.{key}
                  WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *{extra}""")
    return spark.table(f"{schema}.{table}").count()

def stage_bronze():
    wm = spark.sql(f"""SELECT MAX(end_time) AS wm FROM {RUN_LOG}
                       WHERE stage_name = 'Bronze_Ingestion' AND status = 'Success'"""
                   ).collect()[0]["wm"]
    total = 0
    for table, fname in DATASETS:
        cols = BUSINESS_COLUMNS[table]
        spark.sql(f"CREATE TABLE IF NOT EXISTS {BRONZE}.{table} "
                  f"({', '.join(c + ' STRING' for c in cols)}) USING DELTA")
        path = f"{FRESH_LANDING}/{fname}"
        mtime = datetime.datetime.fromtimestamp(
            dbutils.fs.ls(path)[0].modificationTime / 1000.0, tz=datetime.UTC)
        wm_utc = (wm.replace(tzinfo=datetime.UTC)
                  if (wm is not None and wm.tzinfo is None) else wm)
        # DEF-02 hardening: unchanged file AND non-empty bronze table -> skip
        if wm_utc is not None and mtime <= wm_utc and spark.table(f"{BRONZE}.{table}").count() > 0:
            print(f"  bronze.{table}: SKIP (unchanged, table has data)")
            continue
        src_df = read_csv_all_strings(spark, path, cols)   # DEF-01: validates header
        new_df = new_rows_only(src_df, spark.table(f"{BRONZE}.{table}"), cols)
        n = new_df.count()
        if n:
            col_sql = ", ".join(cols)
            new_df.createOrReplaceTempView("v_bronze_new")
            spark.sql(f"INSERT INTO {BRONZE}.{table} ({col_sql}) SELECT {col_sql} FROM v_bronze_new")
            total += n
        print(f"  bronze.{table}: +{n} rows")
    return total

def stage_silver():
    ts = datetime.datetime.now(datetime.UTC).isoformat()
    counts = 0
    counts += _upsert(transform_studies(spark.table(f"{BRONZE}.studies"), ts), SILVER, "studies", "study_id")
    counts += _upsert(transform_sites(spark.table(f"{BRONZE}.sites"), ts), SILVER, "sites", "site_id")
    valid, quar = transform_subjects(spark.table(f"{BRONZE}.subjects"), ts)
    counts += _upsert(valid, SILVER, "subjects", "subject_id")
    counts += _upsert(quar, SILVER, "subjects_quarantine", "subject_id")
    visits, visits_q = transform_visits(spark.table(f"{BRONZE}.visits"), spark.table(f"{SILVER}.subjects"), ts)
    counts += _upsert(visits, SILVER, "visits", "visit_id", delete_not_matched_by_source=True)
    counts += _upsert(visits_q, SILVER, "visits_quarantine", "visit_id")
    checks = run_dq_checks(
        subjects=spark.table(f"{SILVER}.subjects"), visits=spark.table(f"{SILVER}.visits"),
        studies=spark.table(f"{SILVER}.studies"), sites=spark.table(f"{SILVER}.sites"))
    print(summarize(checks))
    if not all_passed(checks):
        raise RuntimeError("Silver DQ gate failed — see DQ summary above")
    return counts

def stage_gold():
    ts = datetime.datetime.now(datetime.UTC).isoformat()
    n1 = _upsert(build_study_kpi_summary(
        spark.table(f"{SILVER}.studies"), spark.table(f"{SILVER}.subjects"),
        spark.table(f"{SILVER}.visits"), gold_ts=ts), GOLD, "study_kpi_summary", "study_id")
    n2 = _upsert(build_site_kpi_summary(
        spark.table(f"{SILVER}.sites"), spark.table(f"{SILVER}.subjects"),
        spark.table(f"{SILVER}.visits"), gold_ts=ts), GOLD, "site_kpi_summary", "site_id")
    return n1 + n2

print("Stage harness ready (DEF-01 header validation + DEF-02 hardened skip active)")

In [0]:
# --- Run the single route Bronze -> Silver -> Gold with stage logging ---
# Reset the isolated fresh environment so every execution of this notebook is
# a genuinely CLEAN end-to-end run from fresh inputs.
for _schema in (BRONZE, SILVER, GOLD):
    spark.sql(f"DROP SCHEMA IF EXISTS {_schema} CASCADE")
    spark.sql(f"CREATE SCHEMA {_schema}")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.freshtest")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {RUN_LOG} (
    run_id STRING, stage_name STRING, start_time TIMESTAMP, end_time TIMESTAMP,
    status STRING, records_processed BIGINT, error_message STRING) USING DELTA""")

EXPECTED = {"Bronze_Ingestion": 1804, "Silver_Transformation": 1796, "Gold_KPI_Analytics": 32}
results = {}
for stage_name, stage_fn in [("Bronze_Ingestion", stage_bronze),
                             ("Silver_Transformation", stage_silver),
                             ("Gold_KPI_Analytics", stage_gold)]:
    start = datetime.datetime.now(datetime.UTC)
    _log_stage(stage_name, start, "Running")
    try:
        records = stage_fn()
        _log_stage(stage_name, start, "Success", records=records)
        results[stage_name] = records
    except Exception as e:
        _log_stage(stage_name, start, "Failed", error=e)
        raise

print("=" * 60)
all_ok = all(results[k] == v for k, v in EXPECTED.items())
print(f"Fresh pipeline run {RUN_ID} complete: {results}")
print(f"Expected: {EXPECTED}")
print("RESULT:", "CLEAN RUN CONFIRMED — all stage records match expectations" if all_ok else "MISMATCH vs expected counts — investigate!")

In [0]:
# --- Confirm the 3 KPIs: (1) vs production gold fingerprints, (2) independent
# recomputation, (3) reconciliation, (4) KPI display, (5) run-log evidence ---
import hashlib

EXPECTED_FINGERPRINTS = {"study_kpi_summary": "ea64730bc6effd53", "site_kpi_summary": "3ddfe6e7889e09e6"}
DROP_FOR_COMPARE = {"study_kpi_summary": ["visits_with_actual_date"],   # prod table predates these cols
                     "site_kpi_summary": ["activation_date"]}

def fingerprint(table, drop_extra):
    pdf = spark.table(table).drop("gold_load_timestamp", *drop_extra).toPandas()
    pdf = pdf.sort_values(list(pdf.columns)).reset_index(drop=True)
    return hashlib.sha256(pdf.to_csv(index=False).encode()).hexdigest()[:16]

print("=== 1) Gold KPI tables vs production baseline (shared columns) ===")
for t, exp_fp in EXPECTED_FINGERPRINTS.items():
    fresh_fp = fingerprint(f"{GOLD}.{t}", DROP_FOR_COMPARE[t])
    base_fp = fingerprint(f"workspace.gold.{t}", [])
    print(f"{t}: fresh={fresh_fp} baseline={base_fp} -> "
          + ("MATCH — expected KPI table reproduced" if fresh_fp == base_fp == exp_fp else "CHECK FAILED"))

print("\n=== 2) Independent KPI recomputation (correlated subqueries, fresh silver) ===")
for name, (sql, key_col, gold_table, gold_col) in independent_kpi_queries(SILVER).items():
    rec = spark.sql(sql).selectExpr(f"id as {key_col}", "val as rec_val")
    stored = spark.table(f"{GOLD}.{gold_table}").select(key_col, gold_col)
    j = rec.join(stored, key_col)
    diff = j.filter(~j["rec_val"].eqNullSafe(j[gold_col])).count()
    missing = rec.join(stored, key_col, "left_anti").count() + stored.join(rec, key_col, "left_anti").count()
    print(f"{name}: value mismatches={diff}, key mismatches={missing} -> "
          + ("PASS" if diff + missing == 0 else "FAIL"))

print("\n=== 3) Reconciliation (fresh schemas) ===")
b_subj, s_subj = spark.table(f"{BRONZE}.subjects").count(), spark.table(f"{SILVER}.subjects").count()
q_subj = spark.table(f"{SILVER}.subjects_quarantine").count()
b_vis, s_vis = spark.table(f"{BRONZE}.visits").count(), spark.table(f"{SILVER}.visits").count()
q_vis = spark.table(f"{SILVER}.visits_quarantine").count()
print(f"bronze.subjects {b_subj} = silver {s_subj} + quarantine {q_subj} + deduped {b_subj - s_subj - q_subj}  -> "
      + ("OK" if b_subj == s_subj + q_subj + 8 else "CHECK"))
print(f"bronze.visits {b_vis} = silver {s_vis} + quarantine {q_vis}  -> "
      + ("OK" if b_vis == s_vis + q_vis else "CHECK"))

print("\n=== 4) The 3 KPIs (study level) ===")
display(spark.table(f"{GOLD}.study_kpi_summary").select(
    "study_id", "enrolled_subjects", "enrollment_rate",
    "visit_completion_rate", "visit_data_completeness"))

print("=== 5) Run-log evidence (this run) ===")
display(spark.sql(f"""SELECT stage_name, status, records_processed, error_message
                     FROM {RUN_LOG} WHERE run_id = '{RUN_ID}' ORDER BY start_time"""))